# AgriVision — Training Model Diagnosis Penyakit (Tomat & Cabai)
### v5 — dataset cabai lebih lengkap (Roboflow, 6 kelas), API key bisa diupload sebagai file

Notebook ini untuk dijalankan di **Google Colab** (GPU gratis, tidak perlu GPU sendiri).

**Dataset yang dipakai (keduanya lewat API resmi, tidak ada download-manual-lalu-upload-ulang):**
- **Tomat:** Kaggle `vipoooool/new-plant-diseases-dataset` — turunan augmented dari PlantVillage, 10 kelas tomat, sudah ter-split train/valid, ~1.4 GB. (Tidak berubah dari versi sebelumnya.)
- **Cabai:** Roboflow Universe `chili-leaves-disease-classification` — 1.147 foto daun cabai, **6 kelas**: healthy, leaf curl, leaf spot, powdery mildew, whitefly, yellowish. Lisensi CC BY 4.0.

**Riwayat lengkap dataset cabai (4 iterasi — didokumentasikan biar jelas kenapa berubah-ubah, bukan asal ganti):**
1. Dua kandidat Kaggle awal ditolak: `shuvokumarbasak4004/...` (cuma 2 kelas kesehatan buah, bukan penyakit daun) dan `ravindubandara3002/...` (struktur tidak bisa diverifikasi).
2. Mendeley `wzc6r6w5w5` (4 kelas, 1.544 gambar, CC BY 4.0) — kontennya bagus, TAPI Mendeley tidak punya API download stabil → upload manual ZIP ke Colab lewat `files.upload()` berulang kali macet/stuck di 0%.
3. Kaggle `dhenyd/chili-plant-disease` (5 kelas, 500 gambar) — solusi sementara, bisa full-API lewat Kaggle, tapi ternyata terlalu kecil.
4. **Roboflow `chili-leaves-disease-classification` (6 kelas, 1.147 gambar) — pilihan final** di notebook ini: lebih besar dari opsi Kaggle sebelumnya, tetap foto close-up daun (bukan foto lapangan/kanopi), tetap ada kelas *healthy*, dan py Roboflow SDK memungkinkan download otomatis lewat API — persis seperti alur Kaggle untuk tomat.

**Kenapa bukan opsi lain yang lebih besar:** ada dataset pepper (bukan cabai/chili) di Hugging Face dengan 26.377 gambar yang bisa didownload tanpa API key sama sekali — tapi itu foto tanaman lapangan/kanopi (bukan close-up daun), **tidak punya kelas sehat**, dan nama kelasnya tidak jelas terkait penyakit cabai spesifik — jadi tidak cocok untuk pipeline diagnosis daun yang konsisten dengan sisi tomat. Dataset cabai terbesar yang benar-benar cocok secara konten (10.987 dan 8.814 gambar, 5-6 kelas termasuk sehat) semuanya ada di Mendeley — sama seperti masalah yang sudah dialami, tidak API-downloadable. Jadi Roboflow adalah titik keseimbangan terbaik saat ini antara ukuran, kecocokan konten, dan otomatisasi penuh.

**PENTING soal "Run All" — realistis, bukan janji berlebihan:** notebook ini menghapus SEMUA langkah download-manual-lalu-upload-ulang yang sebelumnya bikin macet. Yang masih tersisa cuma 2 input kredensial singkat di Step 1 (kirim `kaggle.json` sekali, dan tempel API key Roboflow sekali) — ini BUKAN download manual, cuma memasukkan kunci akses (sama seperti aplikasi mana pun yang butuh login), dan tidak akan "stuck" karena tidak ada file besar yang diupload lewat browser. Setelah kredensial dimasukkan, sisanya (download dataset, training, evaluasi, export) jalan otomatis tanpa interaksi lagi.

**Sebelum mulai:**
1. Buka notebook ini di https://colab.research.google.com (File → Upload notebook)
2. Runtime → Change runtime type → **T4 GPU**
3. Siapkan `kaggle.json` (Kaggle → Settings → API → Create New Token, atau pakai `ml-service/kaggle.json` yang sudah ada)
4. Roboflow API key sudah disiapkan di `ml-service/roboflow_key.txt` di proyekmu — tinggal upload file itu saat diminta. (Kalau perlu bikin baru: daftar gratis di https://app.roboflow.com → Settings → API Keys → salin **Private API Key**.)
5. Klik **Runtime → Run all** — saat diminta, upload `kaggle.json` di satu cell, dan `roboflow_key.txt` di cell berikutnya.

**Sitasi wajib di laporan:**
- Tomat: Mohanty, Hughes & Salathé (2016) — data asli PlantVillage — + dataset Kaggle vipoooool (2018) sebagai sumber praktis.
- Cabai: Roboflow Universe, "Chili leaves disease classification" (2023), CC BY 4.0 — cantumkan link project + versi yang dipakai (lihat `docs/07-referensi.md` yang sudah diperbarui).

In [ ]:
# Cek GPU aktif — kalau kosong/error, ulangi: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install dependencies

In [ ]:
!pip install -q kaggle roboflow
import tensorflow as tf
import os, re, glob, json
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, '__version__') else "bundled with TF (Keras 3)")
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

## Step 1 — Setup Kredensial (Kaggle + Roboflow)

Dua kredensial dibutuhkan — keduanya cuma dimasukkan SEKALI di awal, bukan file besar yang diupload:

1. **Kaggle** — upload `kaggle.json` (tombol Choose Files akan muncul saat cell dijalankan).
2. **Roboflow** — dua cara, pilih salah satu saat cell kedua dijalankan:
   - **Cara A (disarankan, lebih cepat untuk run berikutnya):** upload file `roboflow_key.txt` (berisi API key kamu, satu baris) — sudah ada di folder `ml-service/roboflow_key.txt` di proyekmu, tinggal pilih file itu saat tombol Choose Files muncul.
   - **Cara B:** klik **Cancel upload** pada dialog tersebut, lalu ketik/tempel API key ke kotak yang muncul (`getpass`, karakternya sengaja tidak terlihat di layar — ini normal, bukan error).

In [ ]:
from google.colab import files

uploaded = files.upload()  # pilih file kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API siap.")
print("Kalau nanti muncul error 401/403: token sudah kedaluwarsa — buat token baru di Kaggle, upload ulang, jalankan cell ini lagi.")

In [ ]:
import getpass

print("Upload 'roboflow_key.txt' (dari folder ml-service/ proyekmu) kalau ada — atau klik Cancel upload untuk ketik manual.")
uploaded_key = files.upload()

if uploaded_key:
    key_filename = list(uploaded_key.keys())[0]
    with open(key_filename) as f:
        ROBOFLOW_API_KEY = f.read().strip()
    print(f"Roboflow API key dibaca dari file '{key_filename}'.")
else:
    ROBOFLOW_API_KEY = getpass.getpass("Tempel Roboflow Private API Key kamu (dari app.roboflow.com > Settings > API Keys), lalu Enter: ")

print("Roboflow API key siap dipakai (panjang:", len(ROBOFLOW_API_KEY), "karakter).")

## Step 2 — Download & Siapkan Dataset Tomat

Dataset: **New Plant Diseases Dataset** — https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset

**Catatan terverifikasi:** hasil unzip punya folder ganda bersarang (`New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train`) — ini memang bawaan dataset-nya, bukan bug. Cell di bawah mendeteksi otomatis lewat `glob`, tidak mengasumsikan path secara buta.

In [ ]:
!mkdir -p /content/data/tomato_raw
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/data/tomato_raw --unzip

train_dirs = glob.glob('/content/data/tomato_raw/**/train', recursive=True)
valid_dirs = glob.glob('/content/data/tomato_raw/**/valid', recursive=True)
assert train_dirs and valid_dirs, "train/valid folder tidak ditemukan — cek hasil unzip manual dengan !find /content/data/tomato_raw -maxdepth 4"

TOMATO_TRAIN_SRC = train_dirs[0]
TOMATO_VALID_SRC = valid_dirs[0]
print("train:", TOMATO_TRAIN_SRC)
print("valid:", TOMATO_VALID_SRC)

In [ ]:
# 10 kelas tomat — string PERSIS sesuai dataset (jangan diubah, termasuk spasi & kapitalisasi)
TOMATO_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]

found_train = sorted(c for c in os.listdir(TOMATO_TRAIN_SRC) if c.startswith("Tomato"))
missing = set(TOMATO_CLASSES) - set(found_train)
extra = set(found_train) - set(TOMATO_CLASSES)
if missing:
    print("PERINGATAN — kelas yang diharapkan tapi tidak ditemukan:", missing)
if extra:
    print("Info — kelas tomat tambahan yang ditemukan (di luar daftar 10 di atas):", extra)
if not missing:
    print(f"OK — semua {len(TOMATO_CLASSES)} kelas tomat yang diharapkan ditemukan.")

In [ ]:
import shutil

TOMATO_TRAIN = '/content/data/tomato/train'
TOMATO_VALID = '/content/data/tomato/valid'
os.makedirs(TOMATO_TRAIN, exist_ok=True)
os.makedirs(TOMATO_VALID, exist_ok=True)

def copy_tomato_classes(src, dst):
    for c in TOMATO_CLASSES:
        s, d = os.path.join(src, c), os.path.join(dst, c)
        if os.path.exists(s) and not os.path.exists(d):
            shutil.copytree(s, d)

copy_tomato_classes(TOMATO_TRAIN_SRC, TOMATO_TRAIN)
copy_tomato_classes(TOMATO_VALID_SRC, TOMATO_VALID)
print("Selesai menyalin kelas tomat ke", TOMATO_TRAIN, "dan", TOMATO_VALID)

## Step 3 — Download & Siapkan Dataset Cabai (Roboflow)

Project: **Chili leaves disease classification** — https://universe.roboflow.com/chili-leaves-disease-classification/chili-leaves-disease-classification

**Catatan ketidakpastian yang jujur:** nomor versi dataset (`version(1)`) di bawah adalah tebakan berdasar riset — kalau project sudah punya versi lebih baru, sel ini akan gagal atau menarik versi lama. Kalau errornya soal "version not found", buka link project di atas → tab **Versions** → cek nomor versi terbaru → ganti angka `1` di `project.version(1)` sesuai itu.

Format download `"folder"` biasanya menghasilkan struktur `train/valid/test` dengan subfolder per kelas (mirip dataset tomat) — tapi kalau ternyata flat (langsung subfolder kelas tanpa split), cell berikutnya otomatis mendeteksi dan menangani kedua kemungkinan.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("chili-leaves-disease-classification").project("chili-leaves-disease-classification")
version = project.version(1)  # <-- cek tab "Versions" di halaman project kalau ini gagal
chili_dataset = version.download("folder", location="/content/data/chili_raw")
print("Download selesai ke:", chili_dataset.location)

In [ ]:
# Deteksi struktur hasil download: sudah ter-split (train/valid/test) atau flat (subfolder kelas langsung)?
CHILI_ROOT_BASE = '/content/data/chili_raw'

chili_train_dirs = glob.glob(f'{CHILI_ROOT_BASE}/**/train', recursive=True)
chili_valid_dirs = glob.glob(f'{CHILI_ROOT_BASE}/**/valid', recursive=True)

CHILI_PRESPLIT = bool(chili_train_dirs and chili_valid_dirs)

if CHILI_PRESPLIT:
    CHILI_TRAIN_SRC = chili_train_dirs[0]
    CHILI_VALID_SRC = chili_valid_dirs[0]
    print("Dataset cabai SUDAH ter-split. train:", CHILI_TRAIN_SRC, "| valid:", CHILI_VALID_SRC)
    _scan_root = CHILI_TRAIN_SRC
else:
    # Flat: cari folder yang isinya beberapa subfolder kelas
    candidate_roots = []
    for d in glob.glob(f'{CHILI_ROOT_BASE}/**/', recursive=True):
        subdirs = [s for s in os.listdir(d) if os.path.isdir(os.path.join(d, s))]
        if len(subdirs) >= 3:
            candidate_roots.append((d, sorted(subdirs)))
    assert candidate_roots, "Tidak ditemukan folder train/valid ATAUPUN folder berisi >=3 subfolder kelas — cek manual: !find /content/data/chili_raw -maxdepth 4"
    CHILI_ROOT = max(candidate_roots, key=lambda t: len(t[1]))[0]
    print("Dataset cabai BELUM ter-split (flat). CHILI_ROOT:", CHILI_ROOT)
    _scan_root = CHILI_ROOT

In [ ]:
# Normalisasi nama untuk pencocokan fleksibel (case-insensitive, spasi==underscore)
def _norm(s):
    return re.sub(r'[\s_]+', ' ', s.strip().lower())

CHILI_CLASSES_EXPECTED = ["healthy", "leaf curl", "leaf spot", "powdery mildew", "whitefly", "yellowish"]

found_chili = sorted(c for c in os.listdir(_scan_root) if os.path.isdir(os.path.join(_scan_root, c)))
found_norm = {_norm(c): c for c in found_chili}
expected_norm = set(CHILI_CLASSES_EXPECTED)

missing = expected_norm - set(found_norm.keys())
extra = set(found_norm.keys()) - expected_norm

print("Kelas cabai ditemukan (nama folder asli):", found_chili)
if missing:
    print("PERINGATAN — kelas yang diharapkan tapi tidak ditemukan:", missing)
    print("-> Kemungkinan versi dataset berbeda dari yang diasumsikan. Cek manual: !find", _scan_root, "-maxdepth 2")
if extra:
    print("Info — folder tambahan di luar 6 kelas yang diharapkan:", extra)
if not missing:
    print(f"OK — semua {len(CHILI_CLASSES_EXPECTED)} kelas cabai yang diharapkan ditemukan.")

for c in found_chili:
    n = len([f for f in os.listdir(os.path.join(_scan_root, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f" - {c}: {n} gambar")

## Step 4 — Data Pipeline (augmentasi)

Dataset tomat sudah ter-split (train/valid terpisah). Dataset cabai **tergantung hasil deteksi Step 3** — kalau Roboflow sudah menyediakan train/valid, dipakai langsung; kalau belum, dipisah otomatis 80/20 lewat `validation_split`.

Karena dataset cabai masih relatif kecil (±190 gambar/kelas), augmentasi tetap penting untuk mengurangi overfitting — jangan dihapus/dikurangi tanpa alasan kuat.

In [ ]:
IMG_SIZE_MOBILENET = (224, 224)
IMG_SIZE_EFFICIENTNET = (380, 380)
BATCH_SIZE = 32

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

def prep_train(ds):
    ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def prep_val(ds):
    return ds.prefetch(tf.data.AUTOTUNE)

# --- Tomat: sudah ter-split ---
tomato_train_ds = tf.keras.utils.image_dataset_from_directory(
    TOMATO_TRAIN, image_size=IMG_SIZE_MOBILENET, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=42
)
tomato_class_names = tomato_train_ds.class_names
tomato_valid_ds = tf.keras.utils.image_dataset_from_directory(
    TOMATO_VALID, image_size=IMG_SIZE_MOBILENET, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
)
tomato_train_ds = prep_train(tomato_train_ds)
tomato_valid_ds = prep_val(tomato_valid_ds)
print("Kelas tomat (urutan index model):", tomato_class_names)

In [ ]:
# --- Cabai: pakai train/valid Roboflow kalau ada, kalau tidak pakai validation_split ---
if CHILI_PRESPLIT:
    chili_train_ds = tf.keras.utils.image_dataset_from_directory(
        CHILI_TRAIN_SRC, image_size=IMG_SIZE_EFFICIENTNET, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=42
    )
    chili_class_names = chili_train_ds.class_names
    chili_valid_ds = tf.keras.utils.image_dataset_from_directory(
        CHILI_VALID_SRC, image_size=IMG_SIZE_EFFICIENTNET, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
    )
else:
    chili_train_ds = tf.keras.utils.image_dataset_from_directory(
        CHILI_ROOT, validation_split=0.2, subset="training", seed=123,
        image_size=IMG_SIZE_EFFICIENTNET, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    chili_class_names = chili_train_ds.class_names
    chili_valid_ds = tf.keras.utils.image_dataset_from_directory(
        CHILI_ROOT, validation_split=0.2, subset="validation", seed=123,
        image_size=IMG_SIZE_EFFICIENTNET, batch_size=BATCH_SIZE, label_mode='categorical'
    )

chili_train_ds = prep_train(chili_train_ds)
chili_valid_ds = prep_val(chili_valid_ds)
print("Kelas cabai (urutan index model):", chili_class_names)

## Step 5 — Model Builder (Transfer Learning)

Sesuai `docs/05-ml-pipeline.md`: **MobileNetV2** untuk tomat (224×224), **EfficientNetB4** untuk cabai (380×380).

**Preprocessing per arsitektur (terverifikasi, sering jadi sumber bug diam-diam):**
- MobileNetV2 → `preprocess_input` menskalakan ke [-1, 1] — input harus raw 0–255, JANGAN dibagi 255 manual duluan.
- EfficientNetB4 → normalisasi sudah built-in di dalam model, `preprocess_input`-nya cuma pass-through — input juga harus raw 0–255, JANGAN dibagi 255 manual.

Kedua fungsi builder di bawah sudah menangani ini dengan benar (preprocessing dipanggil di dalam graph model, bukan di data pipeline), jadi tinggal pakai — tidak perlu rescaling tambahan di mana pun.

In [ ]:
def build_model(base_arch, num_classes, img_size):
    input_shape = img_size + (3,)
    inputs = tf.keras.Input(shape=input_shape)

    if base_arch == "mobilenet_v2":
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        base_model = tf.keras.applications.MobileNetV2(
            input_shape=input_shape, include_top=False, weights="imagenet"
        )
    elif base_arch == "efficientnet_b4":
        preprocess = tf.keras.applications.efficientnet.preprocess_input
        base_model = tf.keras.applications.EfficientNetB4(
            input_shape=input_shape, include_top=False, weights="imagenet"
        )
    else:
        raise ValueError("base_arch tidak dikenal")

    base_model.trainable = False  # freeze dulu untuk tahap awal training

    x = preprocess(inputs)          # raw 0-255 masuk sini, JANGAN di-rescale sebelum ini
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

## Step 6 — Training Model Tomat (MobileNetV2)

In [ ]:
tomato_model, tomato_base = build_model("mobilenet_v2", len(tomato_class_names), IMG_SIZE_MOBILENET)

tomato_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_tomato = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ModelCheckpoint("/content/tomato_best.keras", save_best_only=True, monitor="val_accuracy"),
]

history_tomato = tomato_model.fit(
    tomato_train_ds, validation_data=tomato_valid_ds, epochs=15, callbacks=callbacks_tomato
)

In [ ]:
# Fine-tuning tahap 2: unfreeze sebagian layer atas base model, lanjut training dengan LR kecil
tomato_base.trainable = True
for layer in tomato_base.layers[:-30]:
    layer.trainable = False

tomato_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_tomato_ft = tomato_model.fit(
    tomato_train_ds, validation_data=tomato_valid_ds, epochs=8, callbacks=callbacks_tomato
)

## Step 7 — Training Model Cabai (EfficientNetB4)

**Catatan penting:** dataset cabai (1.147 gambar, 6 kelas — sekitar 150 gambar training per kelas) masih jauh lebih kecil dibanding tomat (~87.900 gambar, 10 kelas), walau sudah lebih besar dari opsi Kaggle sebelumnya (500 gambar). Wajar kalau akurasi validasi cabai lebih fluktuatif dan lebih cepat overfit dibanding tomat — laporkan angka apa adanya di bab evaluasi, dan diskusikan gap ini secara eksplisit (sama seperti gap lab-vs-lapangan untuk tomat).

In [ ]:
chili_model, chili_base = build_model("efficientnet_b4", len(chili_class_names), IMG_SIZE_EFFICIENTNET)

chili_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_chili = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ModelCheckpoint("/content/chili_best.keras", save_best_only=True, monitor="val_accuracy"),
]

history_chili = chili_model.fit(
    chili_train_ds, validation_data=chili_valid_ds, epochs=25, callbacks=callbacks_chili
)

In [ ]:
chili_base.trainable = True
for layer in chili_base.layers[:-30]:
    layer.trainable = False

chili_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_chili_ft = chili_model.fit(
    chili_train_ds, validation_data=chili_valid_ds, epochs=10, callbacks=callbacks_chili
)

## Step 8 — Evaluasi (Confusion Matrix, Precision/Recall/F1)

**Untuk laporan TA:** ini evaluasi pada data uji dari dataset yang sama. Sesuai `docs/05-ml-pipeline.md`, idealnya juga uji dengan foto lapangan tambahan (PlantDoc untuk tomat, atau foto sendiri untuk cabai) dan laporkan hasilnya terpisah dari angka di bawah ini.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(model, valid_ds, class_names, title):
    y_true, y_pred = [], []
    for x, y in valid_ds:
        preds = model.predict(x, verbose=0)
        y_true.extend(np.argmax(y.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    print(f"=== {title} ===")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(6, len(class_names)), max(5, len(class_names) * 0.8)))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Greens')
    plt.title(f"Confusion Matrix — {title}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f"/content/confusion_matrix_{title.lower()}.png", dpi=150)
    plt.show()

evaluate_model(tomato_model, tomato_valid_ds, tomato_class_names, "Tomat")

In [ ]:
evaluate_model(chili_model, chili_valid_ds, chili_class_names, "Cabai")

## Step 9 — Export ke TFLite + Label Map

`label_map.json` berisi mapping index model → nama kelas asli → `disease_label` (snake_case) yang dipakai FastAPI service dan harus dicocokkan dengan tabel `disease_reference` di `database/schema.sql`.

In [ ]:
def export_tflite(model, path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(path, "wb") as f:
        f.write(tflite_model)
    print(f"Tersimpan: {path} ({len(tflite_model)/1e6:.2f} MB)")

export_tflite(tomato_model, "/content/tomato_classifier.tflite")
export_tflite(chili_model, "/content/chili_classifier.tflite")

In [ ]:
def slugify(name):
    s = re.sub(r'[^a-zA-Z0-9]+', '_', name.strip())
    return s.strip('_').lower()

def to_disease_label(crop, raw_class_name):
    s = slugify(raw_class_name)
    if s.startswith('healthy'):
        return f"{crop}_healthy"
    if s.startswith(crop + '_'):
        return s
    return f"{crop}_{s}"

label_map = {
    "tomato": {
        "model_version": "v1",
        "img_size": list(IMG_SIZE_MOBILENET),
        "source_dataset": "kaggle:vipoooool/new-plant-diseases-dataset",
        "classes": [
            {"index": i, "raw_class_name": name, "disease_label": to_disease_label("tomato", name)}
            for i, name in enumerate(tomato_class_names)
        ],
    },
    "chili": {
        "model_version": "v1",
        "img_size": list(IMG_SIZE_EFFICIENTNET),
        "source_dataset": "roboflow:chili-leaves-disease-classification/chili-leaves-disease-classification (v1 — verifikasi nomor versi asli di Step 3)",
        "classes": [
            {"index": i, "raw_class_name": name, "disease_label": to_disease_label("chili", name)}
            for i, name in enumerate(chili_class_names)
        ],
    },
}

with open("/content/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2, ensure_ascii=False)

print(json.dumps(label_map, indent=2, ensure_ascii=False))

**PENTING — langkah lanjutan setelah ini (bukan otomatis):**

`disease_label` di atas kemungkinan besar TIDAK semuanya sudah ada di `database/schema.sql`. Kelas yang perlu ada (dari dataset yang dipakai notebook ini):

- **Tomat (10):** `tomato_bacterial_spot`, `tomato_early_blight`, `tomato_late_blight`, `tomato_leaf_mold`, `tomato_septoria_leaf_spot`, `tomato_spider_mites_two_spotted_spider_mite`, `tomato_target_spot`, `tomato_tomato_yellow_leaf_curl_virus`, `tomato_tomato_mosaic_virus`, `tomato_healthy`
- **Cabai (6):** `chili_healthy`, `chili_leaf_curl`, `chili_leaf_spot`, `chili_powdery_mildew`, `chili_whitefly`, `chili_yellowish`

`database/schema.sql` yang ada di proyekmu sudah diperbarui untuk mencakup semua ini — cek lagi setelah training selesai, cocokkan `disease_label` yang dicetak di atas dengan yang ada di `schema.sql` (harusnya sudah pas, tapi selalu baik untuk verifikasi sekali lagi kalau nanti Roboflow merilis versi dataset baru dengan kelas berbeda).

Isi rekomendasi (`recommendation_text`, `dosage`, dst) yang ditandai `[PLACEHOLDER]` di `schema.sql` **wajib diverifikasi ke dosen pembimbing atau sumber agronomi terpercaya** sebelum dipakai di laporan final — dosis pestisida yang salah berisiko nyata.

In [ ]:
# Cell terakhir: bundling semua hasil jadi satu zip untuk didownload
!zip -j /content/agrivision_models.zip \
    /content/tomato_classifier.tflite \
    /content/chili_classifier.tflite \
    /content/label_map.json \
    /content/confusion_matrix_tomat.png \
    /content/confusion_matrix_cabai.png

files.download('/content/agrivision_models.zip')

## Step 10 — Setelah Download

1. Extract `agrivision_models.zip`.
2. Pindahkan `tomato_classifier.tflite`, `chili_classifier.tflite`, `label_map.json` ke folder `ml-service/models/` di proyek lokal kamu.
3. Simpan kedua gambar confusion matrix ke lampiran laporan — bukti evaluasi untuk bab hasil.
4. Cocokkan `disease_label` di `label_map.json` dengan `database/schema.sql` (`disease_reference`) — sudah disiapkan, tapi verifikasi lagi terutama kalau ada kelas baru yang tidak terduga.
5. Cantumkan sitasi Roboflow + nomor versi dataset yang benar-benar kepakai (dicetak di Step 3/9) di laporan.
6. Lanjutkan ke langkah 3 di `CLAUDE.md`: bangun FastAPI service yang memuat model ini sesuai `docs/04-api-contract.md`.